In [14]:
from scapy.all import Ether, sendp, get_if_list
import time

# List available network interfaces
interfaces = get_if_list()
print("Available interfaces:")
for iface in interfaces:
    print(iface)


Available interfaces:
\Device\NPF_{02741F9E-884C-4A62-BE45-423AFA520097}
\Device\NPF_{9AC1E72B-7F72-49E7-850C-4F18896F94CA}
\Device\NPF_{0D10BCB1-6E31-4BB9-AA69-682E92525E9B}
\Device\NPF_{88214AF4-0292-4167-A00F-E0990C869E72}
\Device\NPF_{3C9FD1E2-417E-4DA9-9612-8AC735F01823}
\Device\NPF_{A5707401-5579-44A5-AD24-63A3DCCEF346}
\Device\NPF_{684E3EFC-41CB-4FF3-B971-7466DB7D2532}
\Device\NPF_{C8A62220-B9A0-4ACF-A8D2-E9EC414D3743}
\Device\NPF_{E960F3D2-72F5-4DF1-8282-340C0EEB528A}
\Device\NPF_{37217669-42DA-4657-A55B-0D995D328250}
\Device\NPF_Loopback


In [17]:
# Define the Ethernet frame
eth_frame = Ether(dst="12:34:ff:ff:ff:ff", type=0x8001)

# Send the packet on specified interface (replace 'YOUR_INTERFACE' with the actual interface name from the list above)
#sendp(eth_frame, iface='Ethernet 6', verbose=False)


start_time = time.time()
for _ in range(1000000):  # Adjust the range based on the desired duration
    sendp(eth_frame, iface='Ethernet 6', verbose=False)
    if time.time() - start_time >= 30:  # Run for 10 seconds
        break

In [ ]:
from scapy.all import *
import time

# Packet configuration
packet = IP(dst="target.ip")/UDP(dport=12345)

# Start sending packets


In [55]:
import serial
import numpy as np
import time
import re

def send_command_and_read(serial_port, frame_count, delay_time):
    # Build the command to send to the serial port
    command = f"{frame_count} {delay_time}"
    serial_port.write(command.encode('utf-8'))  # Send command to the serial port
    #time.sleep(delay_time / 1_000_000)  # Convert microseconds to seconds for the delay


    # Regular expression to capture frame number and timing
    pattern = re.compile(r'Frame\[(\d+)\]: (\d+) us')

    # Read data from the serial port
    data = []
    last_time = time.time()  # Store the time of the last read
    while True:
        if serial_port.in_waiting > 0:
            response = serial_port.readline().decode('utf-8').strip()  # Read a line from the serial port
            match = pattern.match(response)  # Match the response against the regex pattern
            if match:
                frame_number = int(match.group(1))  # Extract the frame number
                timing_value = int(match.group(2))   # Extract the timing in microseconds
                data.append((frame_number, timing_value))  # Append as a tuple
                last_time = time.time()  # Update last read time
        # Check if 1 second has passed since the last read
        if time.time() - last_time > 2:
            break  # Exit the loop if no new messages for 1 second

    return np.array(data)  # Convert the list of tuples into a numpy array

def run():
    # User input in the specified format
    #user_input = input("Enter frameCount and delayTimeMicroseconds: ")
    #frame_count, delay_time = map(int, user_input.split())
    frame_count, delay_time = (5000, 50)
    # Set up the serial port (adjust COM port and baud rate as needed)
    with serial.Serial('COM6', 115200, timeout=1) as ser:
        time.sleep(2)  # Allow time for the serial connection to initialize
        data_array = send_command_and_read(ser, frame_count, delay_time)
        return data_array

data = run()

In [56]:
data

array([[   0,   53],
       [   1,   52],
       [   2,   53],
       ...,
       [1349,   52],
       [1350,   52],
       [1351,   53]])

In [18]:
US_TO_S = (10**-6)
M100 = 100_000_000
78/(US_TO_S*20)

3900000.0000000005

In [25]:
(500/M100)/US_TO_S

5.000000000000001

In [63]:
1/(100000000/(1514*8))

0.00012111999999999999

In [2]:
import numpy as np
import re
import plotly.graph_objects as go

def parse_log(log_text, use_interval=False):
    # Split the log text into lines
    lines = log_text.strip().split('\n')
    tests = []
    current_test = None
    intervals = []  # List to store interval values
    frame_sizes = []  # List to store frame sizes
    total_tx_times = []  # List to store total Tx times
    total_rx_times = []  # List to store total Rx times

    # Define regex patterns
    frame_pattern = re.compile(r'Frame\[(\d+)\]:\s*(\d+)\s*us')
    size_pattern = re.compile(r'Running test with frame size:\s*(\d+)\s*bytes')
    interval_pattern = re.compile(r'Running test with interval:\s*(\d+)\s*us')
    total_time_pattern = re.compile(r'Total Tx time:\s*(\d+)\s*us,\s*Total Rx time:\s*(\d+)\s*us')

    for line in lines:
        # Check if the log line specifies frame size or interval
        if use_interval:
            interval_match = interval_pattern.search(line)
            if interval_match:
                interval = int(interval_match.group(1))  # Get the interval
                intervals.append(interval)  # Store the interval
                if current_test is not None:
                    tests.append(np.array(current_test))
                current_test = []  # Start a new test
        else:  # Frame size or other case
            size_match = size_pattern.search(line)
            if size_match:
                frame_size = int(size_match.group(1))  # Get the frame size
                frame_sizes.append(frame_size)  # Store the frame size
                if current_test is not None:
                    tests.append(np.array(current_test))
                current_test = []  # Start a new test

        # Match frame lines using regex
        frame_match = frame_pattern.search(line)
        if frame_match:
            index = int(frame_match.group(1))  # Frame index
            us_value = int(frame_match.group(2))  # us value
            current_test.append((index, us_value))
        
        # Match total Tx and Rx time line
        total_time_match = total_time_pattern.search(line)
        if total_time_match:
            total_tx_time = int(total_time_match.group(1))  # Total Tx time
            total_rx_time = int(total_time_match.group(2))  # Total Rx time
            total_tx_times.append(total_tx_time)
            total_rx_times.append(total_rx_time)

    # Append the last test if it exists
    if current_test is not None:
        tests.append(np.array(current_test))

    return tests, frame_sizes, total_tx_times, total_rx_times, intervals


def read_log_file(file_path):
    with open(file_path, 'r') as log_file:
        log_text = log_file.read()
    return log_text

def read_parse_log(file_path, use_interval=False):
    return parse_log(read_log_file(file_path), use_interval=use_interval)

# Specify the log file path
log_file_path = "src/raw_packetsize_sweep_latency_with_cpu_time.txt"

# Read the log from the file
log_content = read_log_file(log_file_path)

# Parse the log
parsed_tests = parse_log(log_content)



In [3]:
(
    raw_packetsize_sweep_latencies, 
    raw_packetsize_sweep_frame_sizes, 
    raw_packetsize_sweep_total_tx_times, 
    raw_packetsize_sweep_total_rx_times, 
    _
    ) = read_parse_log("src/raw_packetsize_sweep_latency_with_cpu_time.txt")
raw_packetsize_sweep_avg_latencies=[np.mean(run[1:,1]) for run in raw_packetsize_sweep_latencies]

(
    raw_interval_sweep_latencies, 
    _, 
    raw_interval_sweep_total_tx_times, 
    raw_interval_sweep_total_rx_times, 
    raw_interval_sweep_intervals
    ) = read_parse_log("src/raw_interval_sweep_latency.txt", use_interval=True)
raw_interval_sweep_avg_latencies=[np.mean(run[1:,1]) for run in raw_interval_sweep_latencies]

(
    udp_packetsize_sweep_latencies, 
    udp_packetsize_sweep_frame_sizes, 
    udp_packetsize_sweep_total_tx_times, 
    udp_packetsize_sweep_total_rx_times, 
    _
    ) = read_parse_log("src/udp_packetsize_sweep_latency_with_cpu_time.txt")
udp_packetsize_sweep_avg_latencies=[np.mean(run[1:,1]) for run in udp_packetsize_sweep_latencies]

(
    udp_interval_sweep_latencies, 
    _, 
    udp_interval_sweep_total_tx_times, 
    udp_interval_sweep_total_rx_times, 
    udp_interval_sweep_intervals
    ) = read_parse_log("src/udp_interval_sweep_latency.txt", use_interval=True)
udp_interval_sweep_avg_latencies=[np.mean(run[1:,1]) for run in udp_interval_sweep_latencies]

In [4]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.array(raw_packetsize_sweep_frame_sizes),
    y=raw_packetsize_sweep_avg_latencies,
    mode='lines+markers',
    name='Raw Frames',
    line=dict(shape='linear')
))

fig.add_trace(go.Scatter(
    x=np.array(udp_packetsize_sweep_frame_sizes),
    y=udp_packetsize_sweep_avg_latencies,
    mode='lines+markers',
    name='UDP',
    line=dict(shape='linear')
))

fig.update_layout(
    title='Average Latency vs Frame Size',
    xaxis_title='Data Size (bytes)',
    yaxis_title='Average Time (us)',
    showlegend=True
)

fig.show()

In [5]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.array(raw_interval_sweep_intervals),
    y=raw_interval_sweep_avg_latencies,
    mode='lines+markers',
    name='Raw Frames',
    line=dict(shape='linear')
))

fig.add_trace(go.Scatter(
    x=np.array(udp_interval_sweep_intervals),
    y=udp_interval_sweep_avg_latencies,
    mode='lines+markers',
    name='UDP',
    line=dict(shape='linear')
))

fig.update_layout(
    title='Average Latency vs Transmission Period',
    xaxis_title='Data Size (bytes)',
    yaxis_title='Average Time (us)',
    showlegend=True
)

fig.show()

In [8]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=np.array(raw_packetsize_sweep_frame_sizes),
    y=np.array(raw_packetsize_sweep_total_tx_times)/10000,
    mode='lines+markers',
    name='Raw Frames',
    line=dict(shape='linear')
))

fig.add_trace(go.Scatter(
    x=np.array(udp_packetsize_sweep_frame_sizes),
    y=np.array(udp_packetsize_sweep_total_tx_times)/10000,
    mode='lines+markers',
    name='UDP',
    line=dict(shape='linear')
))

fig.update_layout(
    title='Transmit CPU time',
    xaxis_title='Data Size (bytes)',
    yaxis_title='Average Time (us)',
    showlegend=True
)

fig.show()